# S6E10 — 15 TabM on the rich feature set

> **Pipeline position:** 10 RealMLP-rich → 12–14 (closed experiments) → **15 TabM** → 11 stack

| | |
|---|---|
| **Input** | `X_rich_*.parquet` (09), `y_train.parquet`; OOF files of the current stack members in `preds/` (for the fold-0 mini-stack) |
| **Output** | `preds/tabm_folds[_s<seed>]/` (fold checkpoints); if accepted: `preds/oof_tabm*.npy`, `test_tabm*.npy` |
| **Runtime** | see the timing probe (RTX 3070 Ti) |
| **Result** | see *Results & decisions* |

**Why TabM:** the stack is carried by models of *different nature* (TabPFN, RealMLP); the GBDTs are near-copies of each other. TabM (Gorishniy et al., 2024) is a different kind of network: **k = 32 MLPs that share most weights** (BatchEnsemble-style adapters) and are trained in parallel, each on its own batch order; the prediction is the mean of the 32. Gives the effect of an ensemble at the cost of roughly one network. It was the strongest MLP-type model in recent tabular benchmarks.

**Configuration (variant B, chosen):** pytabkit defaults of `TabM_D_Classifier` (k = 32, 512-wide blocks, lr 2e-3, dropout 0.1, quantile preprocessing) **plus** piecewise-linear embeddings of numeric features (`num_emb_type='pwl'`, 48 bins, d = 16; "TabM†" in the paper), batch 512 and mixed precision (AMP) for speed, at most 40 epochs (a first run with batch 1024 and no cap ran > 35 min on fold 0 without finishing). Input: the same `nn_frame` + in-fold target encoding as RealMLP-rich (`10`).

**Leakage guard — validation set.** TabM in pytabkit always trains with early stopping (patience 16) on a validation set. If the OOF fold were passed as `X_val` (as RealMLP gets it in `10`, where early stopping is off), the *best epoch would be chosen on the OOF rows* and the OOF score would be optimistic. Therefore **no `X_val` is passed**: pytabkit holds out 20% of the fold's *training* part for early stopping. Cost: the network trains on 80% of the rows.

**How it is judged on fold 0 (new model family):** its own AUC is only orientation; what matters is whether it adds to the stack. A **mini-stack on the rows of fold 0**: logistic regression (C = 10) on the logits of the 10 current members, with vs without TabM, evaluated by an inner 5-fold CV over the fold-0 rows (the same nested logic as `11`, inside one fold). Bar: +0.0002 on fold 0 (single-fold noise). If it passes: all 5 folds and the full `11`.

> **Glossary:** *BatchEnsemble* = an ensemble whose members share one weight matrix and differ only by cheap per-member scaling vectors. *Piecewise-linear embedding (PWL)* = a number is encoded by which of 48 quantile bins it falls into and where inside the bin; lets the network model sharp, non-monotonic effects. *AMP* = mixed precision (16-bit arithmetic where safe); faster on GPU. *Early stopping* = stop when the validation score has not improved for `patience` epochs and restore the best epoch.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
from pytabkit import TabM_D_Classifier
warnings.filterwarnings('ignore')

SMOKE = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 0
print('device:', DEVICE, '| seed:', SEED)

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'X_rich_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds'))
X = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
assert list(X.columns) == list(X_test.columns)
print(X.shape, X_test.shape)

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 04:48:26.864000 16208 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


device: cuda | seed: 0


(699635, 92) (299844, 92)


## 1. Input frame (identical to `10`)

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
FULL = pd.concat([X, X_test], ignore_index=True)
TWIN_LEVELS = {t: sorted(FULL[s].astype(int).unique()) for t, s in TWINS.items()}
KEYS = [c for c in X.columns if c.startswith('key_')]
TE_SOURCES = [c for c in ['flight_distance', 'age', 'fd_div10', 'fd_mod1000'] + KEYS if c in X.columns]
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)

def nn_frame(D):
    D = D.drop(columns=['orig_prob'] + KEYS).copy()
    for t, s in TWINS.items():
        D[t] = pd.Categorical(D[s].astype(int), categories=TWIN_LEVELS[t])
    assert not D.isna().any().any()
    return D

# In-fold target encoding: fitted on the fold's TRAINING part only (inner cv=5), applied to validation/test.
def add_te(Xtr, ytr, others):
    te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
    cols = [f'te_{c}' for c in TE_SOURCES]
    tr_te = pd.DataFrame(te.fit_transform(Xtr[TE_SOURCES].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
    outs = [pd.DataFrame(te.transform(o[TE_SOURCES].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
    return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]

## 2. Parameters (variant B)
Only the keys that differ from pytabkit's `TabM_D_Classifier` defaults are set (the defaults are printed for the record). `val_metric_name='1-auc_ovr'`: early stopping on AUC instead of the default classification error.

In [3]:
from pytabkit.models.sklearn.default_params import DefaultParams
print('pytabkit defaults:', DefaultParams.TABM_D_CLASS)
PARAMS = dict(
    device=DEVICE, random_state=0, verbosity=0,
    num_emb_type='pwl', num_emb_n_bins=48, d_embedding=16,     # piecewise-linear embeddings of numeric features ("TabM-dagger")
    batch_size=512, allow_amp=(DEVICE == 'cuda'),             # 512: same epoch time as 1024 here, half the GPU memory (1024 spilled out of 8 GB VRAM)
    n_epochs=40,                                               # cap: ~30 s per epoch on a full fold; patience 16 alone ran > 60 epochs
    val_metric_name='1-auc_ovr',                               # early stopping on AUC
    val_fraction=0.2,                                          # internal hold-out from the fold's TRAINING part (see the leakage note)
)
if SMOKE: PARAMS.update(n_epochs=1)

def fit_predict(tr_idx, va_idx, seed=None, rows=None, params=None):
    seed = SEED if seed is None else seed
    tr_idx = tr_idx if rows is None else tr_idx[:rows]
    Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], X_test])
    Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
    model = TabM_D_Classifier(**{**(params or PARAMS), 'random_state': seed})
    # Deliberately NO X_val: early stopping must not see the OOF rows (it uses 20% of the training part instead).
    model.fit(Xtr, y[tr_idx], cat_col_names=CAT_COLS)
    p_va, p_te = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return p_va, p_te

def run_fold(k, seed=None):
    seed = SEED if seed is None else seed
    d = PREDS_DIR / ('tabm_folds' if seed == 0 else f'tabm_folds_s{seed}')
    fo, ft = d / f'oof_{k}.npy', d / f'test_{k}.npy'
    if fo.exists() and ft.exists(): return np.load(fo), np.load(ft), None
    t0 = time.time(); p_va, p_te = fit_predict(*FOLDS[k], seed=seed, rows=60000 if SMOKE else None)
    if not SMOKE: d.mkdir(exist_ok=True); np.save(fo, p_va); np.save(ft, p_te)
    return p_va, p_te, (time.time() - t0) / 60

pytabkit defaults: {'arch_type': 'tabm', 'tabm_k': 32, 'num_emb_type': 'none', 'num_emb_n_bins': 48, 'batch_size': 256, 'lr': 0.002, 'weight_decay': 0.0, 'n_epochs': 1000000000, 'patience': 16, 'd_embedding': 16, 'd_block': 512, 'n_blocks': 'auto', 'dropout': 0.1, 'compile_model': False, 'allow_amp': False, 'tfms': ['quantile_tabr'], 'gradient_clipping_norm': None}


## 3. Timing probe
2 epochs on 100,000 training rows (no early stop reached), extrapolated to one epoch of a full fold (≈ 448,000 training rows). The number of epochs until early stopping is unknown in advance (best epoch + 16).

In [4]:
tr0, va0 = FOLDS[0]
t0 = time.time()
_ = fit_predict(tr0, va0[:20000], rows=100_000, params={**PARAMS, 'n_epochs': 2})
t_probe = time.time() - t0
per_epoch = t_probe / 2 * (len(tr0) * 0.8 / 80_000)
print(f'probe: {t_probe:.0f} s -> ≈ {per_epoch:.0f} s per epoch on a full fold (upper bound: setup and test prediction included)')
print(f'e.g. 25 epochs ≈ {25 * per_epoch / 60:.0f} min, 40 epochs ≈ {40 * per_epoch / 60:.0f} min')

probe: 25 s -> ≈ 69 s per epoch on a full fold (upper bound: setup and test prediction included)
e.g. 25 epochs ≈ 29 min, 40 epochs ≈ 46 min


## 4. Fold 0

In [5]:
va0 = FOLDS[0][1]
p_va0, p_te0, mins = run_fold(0)
a_tabm = roc_auc_score(y[va0], p_va0)
ref = PREDS_DIR / 'realmlp_rich_refgpu_folds' / 'oof_0.npy'
print(f'fold 0: TabM {a_tabm:.5f}' + ('' if mins is None else f' ({mins:.1f} min)'),
      f'| RealMLP-rich (same machine, seed 0) {roc_auc_score(y[va0], np.load(ref)):.5f}' if ref.exists() else '')

fold 0: TabM 0.96075 | RealMLP-rich (same machine, seed 0) 0.96138


## 5. Mini-stack on the rows of fold 0
The 10 current stack members are taken exactly as `11` takes them (same registry, seed-averaged files when they exist), restricted to the fold-0 rows. Every member's fold-0 prediction comes from a model that never saw those rows, TabM included. A logistic regression (C = 10, as in `11`) is fitted on 4/5 of the fold-0 rows and scored on the remaining 1/5, five times.

In [6]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
def as_logit(o): return lg(o) if (o.min() >= 0 and o.max() <= 1) else o     # same convention as 11
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp',
         'realmlp_rich': 'realmlp_rich', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
for k_, avg in [('realmlp', 'realmlp_avg'), ('realmlp_rich', 'realmlp_rich_avg')]:
    if (PREDS_DIR / f'oof_{avg}.npy').exists(): NAMES[k_] = avg
B = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float))[va0] for s in NAMES.values()])
T = lg(p_va0)[:, None]; yv = y[va0]
print('members:', list(NAMES), '| Spearman TabM vs members:',
      {n: round(pd.Series(T[:, 0]).corr(pd.Series(B[:, i]), method='spearman'), 4) for i, n in enumerate(NAMES)})

rows = []
for i, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(B, yv)):
    r = {'inner fold': i}
    for name, M in [('stack (10)', B), ('stack + TabM', np.hstack([B, T]))]:
        m = LogisticRegression(C=10.0, max_iter=500).fit(M[a], yv[a]); r[name] = roc_auc_score(yv[b], M[b] @ m.coef_[0])
    rows.append(r)
ms = pd.DataFrame(rows); ms['diff'] = ms['stack + TabM'] - ms['stack (10)']
print(ms.round(5).to_string(index=False))
print(f"\nmean: stack {ms['stack (10)'].mean():.5f} | + TabM {ms['stack + TabM'].mean():.5f} | diff {ms['diff'].mean():+.5f} | positive in {(ms['diff'] > 0).sum()}/5 inner folds")
m = LogisticRegression(C=10.0, max_iter=500).fit(np.hstack([B, T]), yv)
print('weights with TabM:', dict(zip(list(NAMES) + ['tabm'], m.coef_[0].round(3))))

members: ['cat_v1', 'cat_v2', 'xgb', 'lgbm_tuned', 'mlp', 'realmlp', 'realmlp_rich', 'tabpfn', 'tabpfn_raw', 'tabicl'] | Spearman TabM vs members: {'cat_v1': np.float64(0.9757), 'cat_v2': np.float64(0.9771), 'xgb': np.float64(0.9737), 'lgbm_tuned': np.float64(0.9738), 'mlp': np.float64(0.962), 'realmlp': np.float64(0.9796), 'realmlp_rich': np.float64(0.9829), 'tabpfn': np.float64(0.9668), 'tabpfn_raw': np.float64(0.9541), 'tabicl': np.float64(0.9144)}


 inner fold  stack (10)  stack + TabM     diff
          0     0.96142       0.96140 -0.00003
          1     0.96041       0.96040 -0.00001
          2     0.96213       0.96213 -0.00000
          3     0.96224       0.96224 -0.00000
          4     0.96342       0.96343  0.00001

mean: stack 0.96193 | + TabM 0.96192 | diff -0.00001 | positive in 1/5 inner folds


weights with TabM: {'cat_v1': np.float64(-0.017), 'cat_v2': np.float64(-0.224), 'xgb': np.float64(-0.018), 'lgbm_tuned': np.float64(0.309), 'mlp': np.float64(-0.13), 'realmlp': np.float64(-0.023), 'realmlp_rich': np.float64(0.429), 'tabpfn': np.float64(0.77), 'tabpfn_raw': np.float64(0.002), 'tabicl': np.float64(-0.104), 'tabm': np.float64(0.021)}


## 6. All 5 folds (only if the fold-0 mini-stack passed)

In [7]:
RUN_ALL = False
if RUN_ALL and not SMOKE:
    oof = np.full(len(X), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        p_va, p_te, mins = run_fold(k)
        oof[va] = p_va; tests.append(p_te)
        print(f'fold {k}: AUC {roc_auc_score(y[va], p_va):.5f}' + ('' if mins is None else f'  ({mins:.1f} min)'), flush=True)
    print(f'\nOOF AUC (TabM): {roc_auc_score(y, oof):.5f}')
    sfx = '' if SEED == 0 else f'_s{SEED}'
    np.save(PREDS_DIR / f'oof_tabm{sfx}.npy', oof); np.save(PREDS_DIR / f'test_tabm{sfx}.npy', np.mean(tests, axis=0))

## Results & decisions

**Fold 0** (RTX 3070 Ti, variant B: pwl embeddings, batch 512, AMP, ≤ 40 epochs, early stopping on 20% of the training part)

| | Fold 0 AUC |
|---|---|
| **TabM** | **0.96075** (10.2 min) |
| RealMLP-rich, same machine, seed 0 | 0.96138 |
| *tuned LightGBM (04), for orientation* | *0.96095* |

**Training curve (internal validation AUC):** 0.9606 after epoch 1, 0.9602 after epochs 2–3, then a steady decline to ≈ 0.945 by epoch 10. The best epoch is the **first**; early stopping restored it and stopped after 17 epochs. TabM memorises this data very fast (likely through the one-hot encoding of the high-cardinality twins, e.g. 3.5k routes). A first run with batch 1024 and no epoch cap spilled out of the 8 GB VRAM and was stopped after 36 min.

**Mini-stack on the fold-0 rows** (LR C = 10, inner 5-fold CV):

| | Stack (10 members) | + TabM | Difference |
|---|---|---|---|
| mean over inner folds | 0.96193 | 0.96192 | **−0.00001** (positive in 1/5) |

TabM's weight in the stack is 0.02; its Spearman correlation with RealMLP-rich is 0.983 — it is mostly a weaker copy of the existing network members.

**Decision: rejected, step closed.** No all-fold run, nothing added to `11`. A regularised variant (lower learning rate, no high-cardinality one-hot twins) could raise TabM's own AUC, but with a stack weight of 0.02 there is little room for it to add new information.


## Glossary
* **TabM** — k MLPs sharing most weights (BatchEnsemble adapters), trained in parallel; prediction = mean over the k members.
* **PWL embedding** — piecewise-linear encoding of a number over quantile bins.
* **Early stopping / patience** — training stops after `patience` epochs without validation improvement; the best epoch is restored.
* **Mini-stack** — logistic regression on members' logits, evaluated by an inner CV on the rows of one outer fold.
* **AMP** — automatic mixed precision on the GPU.